### 앙상블(Ensemble)
- 여러 개의 분류 모델을 조합해서 더 나은 성능을 내는 방법
- Decision Tree 모델을 증가시켜 나온 랜덤포레스트가 대표적임

#### 랜덤포레스트(Random Forest)
##### 랜덤포레스트 훈련방법
- 부트스트랩 샘플 : **중복**을 허용하는 샘플링 방법
- 샘플링 후에 샘플을 복구하고 다시 샘플링 하는 방법입니다.
- 이와 같이 진행하는 이유는 결정트리에서 **과대적합을 방지**할 수 있기 때문이다.
- 각 결정트리에서 나오는 확률의 합을 트리갯수로 나누어서 결정짓는 모델
- 부트스트랩의 샘플링 갯수 : **특성 개수의 제곱근**

In [1]:
import pandas as pd

In [2]:
wine = pd.read_csv("../Data/wine.csv")

In [3]:
# Feature와 Target
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [4]:
# Train , Test
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=42,
        stratify=target
    )

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

In [8]:
rf = RandomForestClassifier(
    n_jobs=-1 # 내PC의 가용 가능한 모든 자원 사용
)
scores = cross_validate(
    rf,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

In [9]:
scores

{'fit_time': array([0.42087626, 0.41951513, 0.42510104, 0.43703628, 0.47845054]),
 'score_time': array([0.05092025, 0.05133891, 0.05714631, 0.05940104, 0.0887568 ]),
 'test_score': array([0.90288462, 0.90288462, 0.88931665, 0.89027911, 0.88450433]),
 'train_score': array([0.99807554, 0.99807554, 0.9983165 , 0.998076  , 0.9983165 ])}

In [15]:
print("Train :", scores['train_score'].mean())
print("Valid :", scores['test_score'].mean())

Train : 0.9981720130385032
Valid : 0.8939738654031242


In [16]:
rf.fit(train_input, train_target)
rf.score(test_input, test_target)

0.8923076923076924

In [17]:
# 중요 Feature
print(rf.feature_importances_)

[0.23337312 0.499072   0.26755487]


---
#### Extra Tree
- 기본적으로 100개의 Tree 사용
- 노드 분할시, 특성의 제곱근의 개수를 사용
- 특성의 선택을 랜덤하게 선택한다
- 특성의 선택이 랜덤하므로 속도는 Random Forest 보다 빠르다

In [18]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1)

scores = cross_validate(
    et,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print("Train :", scores['train_score'].mean())
print("Valid :", scores['test_score'].mean())

Train : 0.9981720130385032
Valid : 0.8918575553416748


---
#### Gradient Boosting
- **가장 유명한 알고리즘 중 하나**
- 경사하강법 처럼 손실함수 사용
- 손실함수를 보고 손실값이 크면 트리를 추가하여 최적의 값을 도출하는 방법
- 'max depth' 는 '3' 으로 제어됨 -> 과대적합 방지
- 단점 : 손실함수를 확인하고 트리를 추가하는 모델이므로 'n_jobs(병렬처리)' 를 할 수 없다.

In [19]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier()

scores = cross_validate(
    gb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print("Train :", scores['train_score'].mean())
print("Valid :", scores['test_score'].mean())

Train : 0.886136193834053
Valid : 0.8714622047827053


---
#### Histogram Gradient Boosting
- 훈련데이터를 256개의 구간으로 분할하여 훈련시키는 방법
- 특성의 범위가 제한되어 있어 빠른 속도를 제공한다
- 제한된 구간이므로 과대적합을 방지한다

In [21]:
# from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier()

scores = cross_validate(
    hgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print("Train :", scores['train_score'].mean())
print("Valid :", scores['test_score'].mean())

Train : 0.9315949163675888
Valid : 0.8754999259643148


> sklearn 에 ensemble 라이브러리를 프로젝트 중 아마 가장 많이 쓰게 될 것 -> 다양한 모델들을 많이 갖고 있기 때문

---
#### LightGBM
- Gradient Boosting 에서 출발한 모델

In [ ]:
# !pip install lightgbm
# -> sklearn.ensemble 라이브러리에, experimental 라이브러리에 등록 안 됨. => Python 라이브러리에는 등록했으나 Google 에 등록 신청 안 된 케이스

In [9]:
#from sklearn.experimental import li... -> 아직 등록 안 된것을 확인 할 수 있음.
from lightgbm import LGBMClassifier
lgb = LGBMClassifier()

scores = cross_validate(
    lgb,
    train_input,
    train_target,
    return_train_score=True,
    n_jobs=-1
)

print("Train :", scores['train_score'].mean())
print("Valid :", scores['test_score'].mean())

Train : 0.934577605325741
Valid : 0.8805047382838529
